# GM Embodied AI — Senior ML/AI Engineer Tech Screen Study Guide

**Format:** Codility, Python + PyTorch + fundamental ML concepts.

**The four flavors of exercise you may see:**
1. Implement a functional piece in a training routine
2. Implement a basic version of a popular ML architecture
3. Improve existing training code (find the bug / find the bottleneck)
4. Implement an algorithmic piece in an ML problem

**Abilities being tested:**
- Read and modify others' code
- Identify and fix suboptimal ML design decisions
- Create hypotheses and test them

**Permission to ask:** You are explicitly allowed to ask the interviewer about syntax. Use this. Don't burn 4 minutes recalling whether it's `dim=-1` or `axis=-1` (it's `dim`).

---

## How to use this notebook

Each section is a self-contained snippet you should be able to **reconstruct from memory in under 3 minutes**. Don't just read — close the cell, retype it, run it. Familiarity with the muscle-memory patterns (Module skeleton, training step, Dataset trio) is what makes the test feel small.

## Interview meta-strategy

- **Talk while you type.** "I'm going to start with the `nn.Module` skeleton, then add the conv block, then verify shape with a dummy tensor." This satisfies the "read and modify" + "create hypotheses" criteria even when your code is mid-flight.
- **Always run a shape sanity check.** Push a `torch.randn(2, 3, 224, 224)` through your module before claiming it works. Catches 80% of bugs.
- **When fixing code, name the bug class out loud.** "This is a CPU↔GPU thrash" or "This is accumulating the autograd graph in a Python list" — that demonstrates Ability #2.
- **Overfit a single batch** as your hypothesis test. If you can't drive loss to ~0 on one batch, the model is broken, not the data.


---
# Legend — acronyms and shape symbols

## Tensor shape symbols
These letters appear in code comments to describe tensor dimensions. Reading shapes fluently is half the battle.

| Symbol | Meaning | Typical context |
|---|---|---|
| `B` | **B**atch size — number of independent examples in a forward pass | every tensor |
| `C` | **C**hannels — feature maps for images, or input/output dims for conv | `(B, C, H, W)` |
| `H` | **H**eight in pixels | `(B, C, H, W)` |
| `W` | **W**idth in pixels | `(B, C, H, W)` |
| `L` | sequence **L**ength — number of tokens, timesteps, or patches | `(B, L, D)` |
| `D` | embedding **D**imension — feature width per token | `(B, L, D)` |
| `N` | generic count — number of boxes, points, samples | detection, point clouds |
| `n_heads` | number of attention heads | multi-head attention |
| `d_head` | per-head dimension; equals `D / n_heads` | multi-head attention |
| `d_k`, `d_v` | key dim, value dim in attention (usually `d_k = d_v = d_head`) | attention math |

**Layout conventions:**
- Vision (Conv2d): `(B, C, H, W)` — channels-first. PyTorch default.
- Sequence (Transformer): `(B, L, D)` with `batch_first=True`. Always use this.
- RNN default (legacy): `(L, B, D)` with `batch_first=False`. Pass `batch_first=True` to override.

## Acronyms used in this notebook

| Acronym | Expansion | One-line meaning |
|---|---|---|
| AMP | Automatic Mixed Precision | Train in fp16/bf16 for ~2x speed; PyTorch handles the precision-sensitive ops automatically |
| BN | Batch Normalization | Normalizes activations across the batch dimension; reduces internal covariate shift |
| LN | Layer Normalization | Normalizes across the feature dim per-sample; default in Transformers |
| GN | Group Normalization | BN alternative that's batch-size independent; common in detection |
| CBR | Conv → BN → ReLU | The workhorse vision block — name your own helper after this |
| CE | Cross-Entropy (loss) | Standard classification loss; combines log-softmax and NLL |
| MSE | Mean Squared Error | Standard regression loss |
| BCE | Binary Cross-Entropy | Two-class CE; use `BCEWithLogitsLoss` for numerical stability |
| MLP | Multi-Layer Perceptron | Stack of fully-connected layers with nonlinearities |
| CNN | Convolutional Neural Network | Vision backbone built from Conv2d blocks |
| RNN | Recurrent Neural Network | Sequential model with hidden state; LSTM/GRU are variants |
| MHA | Multi-Head Attention | The attention mechanism with parallel heads |
| DPA | (Scaled) Dot-Product Attention | The `softmax(QKᵀ/√d)V` formula inside MHA |
| QKV | Query, Key, Value | The three projections in attention |
| ViT | Vision Transformer | Treats images as sequences of patches |
| GAP | Global Average Pooling | `AdaptiveAvgPool2d(1)` — collapses spatial dims to one number per channel |
| GELU | Gaussian Error Linear Unit | Smooth ReLU variant; default activation in modern Transformers |
| SiLU | Sigmoid-weighted Linear Unit | Also called Swish; common in modern vision (EfficientNet, YOLO) |
| IoU | Intersection over Union | Bounding box overlap metric; range [0, 1] |
| NMS | Non-Maximum Suppression | Greedy filter that removes redundant overlapping detections |
| GAP (RL) | Generalized Advantage Estimation | (different GAE — not used in this notebook, just noting) |
| LR | Learning Rate | The optimizer step size |
| WD | Weight Decay | L2 regularization on parameters; coupled in Adam, decoupled in AdamW |
| RL | Reinforcement Learning | Agent learns from reward via interaction |
| PPO | Proximal Policy Optimization | Modern policy gradient algorithm; common in robotics |
| DDP | Distributed Data Parallel | Multi-GPU training via gradient all-reduce |
| H2D / D2H | Host-to-Device / Device-to-Host | CPU↔GPU memory transfer direction |

## Math notation in equations
- $Q, K, V$ — query, key, value matrices in attention
- $d_k$ — the key/query dimension; the $\sqrt{d_k}$ scaling normalizes dot-product magnitude
- $\pi_\theta$ — a policy parameterized by $\theta$; outputs a distribution over actions given a state
- $\gamma$ — discount factor in RL, in $[0, 1)$; weights future rewards


---
# PyTorch vocab — "why this function, here?"

Quick-scan reference for the functions you'll touch most. Each row: **what it does** and **why you'd call it where I called it**.

## The training step trio
| Call | What it does | Why here |
|---|---|---|
| `optimizer.zero_grad(set_to_none=True)` | Sets `.grad` on every tracked parameter back to `None` | PyTorch *accumulates* gradients across calls to `backward()`. Without zeroing, batch N's update would be polluted by batch N-1's gradients. `set_to_none` is faster than zeroing because it deallocates instead of memsetting. |
| `loss.backward()` | Runs autograd backward pass; computes ∂loss/∂param for every leaf tensor with `requires_grad=True`, stores it in `param.grad` | This is where the chain rule actually executes. After this call, every weight has a gradient ready to be applied. |
| `optimizer.step()` | Reads each parameter's `.grad` and applies the update rule (e.g., for SGD: `p -= lr * p.grad`; for Adam: with momentum and adaptive scaling) | This is the only call that *changes weights*. `backward()` only computes gradients; `step()` consumes them. |

## Autograd control
| Call | What it does | Why here |
|---|---|---|
| `with torch.no_grad():` | Context manager that disables autograd graph construction inside the block | Wrap validation / inference. Halves memory (no graph stored) and speeds up forward by skipping bookkeeping. |
| `with torch.inference_mode():` | Stricter, faster `no_grad` — also disables view tracking and version counters | Use for pure inference servers; tensors produced inside cannot later be used in a backward pass. |
| `tensor.detach()` | Returns a new tensor sharing storage but **without** the autograd graph | Use when you want the *value* of an intermediate tensor but not its history (e.g., logging, EMA, target networks in RL). |
| `tensor.requires_grad_(True)` | Flag a leaf tensor as needing gradients | Rare in normal training (parameters get this automatically) but needed for things like input-gradient attacks or custom learnable tensors. |

## Device and memory
| Call | What it does | Why here |
|---|---|---|
| `tensor.to(device, non_blocking=True)` | Copies tensor to target device (CPU↔GPU). `non_blocking=True` allows async H2D when paired with `pin_memory` | Standard way to move batches to the GPU. The `non_blocking` flag lets the next CPU work overlap with the transfer. |
| `tensor.item()` | Returns a Python float/int from a 1-element tensor; **forces a device sync** | Cheap to call once per epoch for logging. Expensive if called every step inside the inner loop (forces GPU→CPU sync). |
| `tensor.cpu()` | Copy tensor to CPU; forces a sync | Use only when you're done with GPU computation on this tensor. Avoid inside training loops. |
| `pin_memory=True` (DataLoader) | Allocates CPU tensors in page-locked memory, enabling async DMA to GPU | Pairs with `non_blocking=True` on `.to(device)` for overlapped transfer. Free throughput win when training on GPU. |
| `tensor.contiguous()` | Returns tensor with C-contiguous memory layout (copying if necessary) | `view()` requires contiguous memory. After `transpose()` or `permute()`, call this before `view()` or just use `reshape()`. |

## Mixed precision (AMP)
| Call | What it does | Why here |
|---|---|---|
| `with autocast(device_type=..., dtype=torch.float16):` | Runs ops inside the block in fp16 where safe; keeps loss-sensitive ops (e.g., reductions) in fp32 | ~2x speedup on Ampere+ GPUs with no accuracy loss for most models. Only wraps **forward + loss**, never backward. |
| `GradScaler()` | Keeps a running scale factor; multiplies the loss before backward to prevent gradient underflow in fp16 | fp16's small range makes tiny gradients round to zero. Scaler bumps them into the representable range, then unscales before the step. |
| `scaler.scale(loss).backward()` | Multiplies loss by the scale factor, then runs backward | Tiny gradients survive as non-zero fp16 values. The scale propagates through chain rule into the gradients. |
| `scaler.unscale_(optimizer)` | Divides the now-populated `.grad`s by the scale factor, in place | Required before *anything* that inspects gradients (e.g., clipping), so you operate on true-magnitude grads. |
| `scaler.step(optimizer)` | Calls `optimizer.step()` only if no inf/NaN found in grads; otherwise skips this batch | Protects against scale-too-high blowups. The skipped batch isn't a bug — it's the safety mechanism. |
| `scaler.update()` | Adjusts the scale factor: doubles it if recent batches were clean, halves it if any had inf/NaN | Auto-tunes the scale to be as large as possible without overflowing. |

## Gradient health
| Call | What it does | Why here |
|---|---|---|
| `torch.nn.utils.clip_grad_norm_(params, max_norm)` | Rescales gradients so their global L2 norm ≤ `max_norm` | Prevents one outlier batch (exploding gradient) from poisoning a checkpoint. Common with RNNs, Transformers, RL. |
| `torch.nn.utils.clip_grad_value_(params, clip_value)` | Clips each gradient element to `[-clip_value, clip_value]` | Cruder than norm clipping; rarely used in modern code but you'll see it in legacy. |

## Mode switches
| Call | What it does | Why here |
|---|---|---|
| `model.train()` | Sets `training=True` on every submodule | Activates Dropout, makes BatchNorm use **batch statistics** and update running stats. |
| `model.eval()` | Sets `training=False` | Disables Dropout, makes BatchNorm use **running statistics**. Forgetting this is the #1 cause of "my val loss looks weird." |

## Module construction
| Call | What it does | Why here |
|---|---|---|
| `super().__init__()` | Calls `nn.Module.__init__` to set up the internal registries | Must be first line in `__init__`. Without it, `self.layer = nn.Linear(...)` won't actually register `layer` as a submodule and its parameters won't be found by the optimizer. |
| `nn.Parameter(tensor)` | Wraps a tensor so `nn.Module` registers it as a learnable parameter | Use when you have a learnable tensor that isn't already inside a built-in layer (e.g., class tokens, learnable positional embeddings). |
| `self.register_buffer("name", tensor)` | Registers a non-learnable tensor that *moves with the module* on `.to(device)` and is saved/loaded with `state_dict` | Use for running stats, fixed positional encodings, anchor boxes — anything that's state but not trained. |

## Common tensor ops you'll write reflexively
| Call | What it does | Why here |
|---|---|---|
| `F.softmax(x, dim=-1)` | Normalizes along `dim` to a probability distribution | Inside attention; for converting logits to probs when you need them (loss functions don't — they expect logits). |
| `F.cross_entropy(logits, targets)` | Combines `log_softmax` + `nll_loss` in a numerically stable way | Use this, never apply softmax manually before CE — that causes numerical instability. |
| `torch.einsum("ij,jk->ik", A, B)` | Sums of products over labeled dims | The swiss army knife for tensor contractions when `@`/`matmul` isn't expressive enough. |
| `tensor.view(*shape)` | Returns a tensor with new shape, **same storage**; requires contiguous memory | Cheaper than reshape, but fails after transpose without `.contiguous()`. |
| `tensor.reshape(*shape)` | Same as view but copies if necessary | Use when you don't know if input is contiguous (e.g., after permute). |
| `tensor.transpose(d0, d1)` | Swaps two dims; produces a non-contiguous view | Common for `(B, L, D) → (B, D, L)` swaps before `Conv1d`. |
| `tensor.permute(*dims)` | Generalizes transpose to arbitrary dim reordering | Common after `view` reshapes in multi-head attention. |
| `tensor.flatten(start_dim, end_dim)` | Collapses a range of dims into one | `x.flatten(1)` after GAP to go from `(B, C, 1, 1) → (B, C)`. |
| `tensor.squeeze(dim)` / `tensor.unsqueeze(dim)` | Removes / adds a size-1 dim | Aligning shapes for broadcasting. |
| `tensor.masked_fill(mask, value)` | Sets elements where mask is True to value | Used in attention to set masked-out positions to `-inf` before softmax. |


## My Words

`GradScaler` - multiplies the loss by a large factor before backpropagation to keep gradients in a safe numerical range, then divides them back down before the optimizer uses them
- when using mixed precision (fp16), gradients can become so small that they round down to zero (numerical underflow)

`set_to_none=True` - instead of setting gradients to `0.0` (which takes up memory and requires a memory read/write operation), this sets the gradient pointers to `None`

`autocast` - since DL models don't need 32-bit floating-point precision for every operation, `autocast` automatically looks at your operations: it runs safe ones (like matrix multiplication) in 16-bit (which doubles throughput and halves memory) and keeps sensitive ones (like softmax or layer norm) in 32-bit to prevent accuracy loss

`torch.nn.utils.clip_grad_norm_(model.parameters(), grad_norm)` - this prevents the "exploding gradients problem
- if a bad batch causes a massive gradient spike, it can permanently ruin the model weights (aka a "poisoned checkpoint")
- this caps the maximum total norm of the gradients

`torch.compile(model)` - uses a Just-in-Time (JIT) compiler (TorchDynamo) to look at your whole model, fuse multiple operations into single GPU kernels, and reduce Python overhead
- its essentially "free speed" for modern GPUs

# To-Do

1. Can you comfortably walk through a model architecture and state the exact tensor shape at every single step?
2. Do you know how to use `.flatten()` and `.transpose()` efficiently without writing custom `view()` or `reshape()` logic?

## Setup

In [ ]:
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# will make DEVICE a string moving forward
DEVICE = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"torch={torch.__version__}  device={DEVICE}")

---
# Focus Area 1 — The Fixer

These are the patterns you scan for the moment they show you someone else's training code. Memorize the **bug class names** so you can call them out.


## 1.1 `loss.item()` vs `loss` — the silent memory leak

**Bug:** appending `loss` (a tensor with the full autograd graph attached) to a Python list. Memory grows every iteration because each tensor keeps a pointer to the entire computation graph.

**Fix:** `loss.item()` returns a Python float and severs the graph. Use it for logging.


In [ ]:
# BAD — graph stays alive, memory grows linearly with steps
losses_bad = []
# losses_bad.append(loss)            # don't

# GOOD — float, no graph
losses_good = []
# losses_good.append(loss.item())    # do

# If you need to keep a tensor but not the graph:
# tensor.detach()      # same storage, no grad
# tensor.detach().cpu()  # also frees GPU memory

## 1.2 `torch.no_grad()` vs `torch.inference_mode()`

**Bug:** running validation inside the training graph. Doubles memory, slows everything down.

**Fix:** wrap eval in `no_grad()`. `inference_mode()` is even stricter (and slightly faster) — disables version counter bumps, so tensors produced under it can't later be used in autograd. Use `inference_mode` for pure inference servers; `no_grad` when in doubt.


In [ ]:
model = nn.Linear(10, 1).to(DEVICE)
x = torch.randn(4, 10, device=DEVICE)

# Validation pattern — always wrap and always .eval()
model.eval()
with torch.no_grad():
    preds = model(x)

# Stricter, faster for pure inference
with torch.inference_mode():
    preds = model(x)

model.train()  # remember to flip back

## 1.3 `optimizer.zero_grad()` — why and when

**Bug:** missing `zero_grad()` means gradients **accumulate** across batches (PyTorch does this on purpose — it's useful for gradient accumulation, but a bug if unintended).

**Order matters:** `zero_grad → forward → loss → backward → step`. Putting `zero_grad` after `step` works too, but the canonical placement is **before forward**.

**Perf tip:** `set_to_none=True` is faster than zeroing — it deallocates instead of memsetting. Default in PyTorch ≥2.0 is already `True`.


In [ ]:
model = nn.Linear(10, 1).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

x = torch.randn(32, 10, device=DEVICE)
y = torch.randn(32, 1, device=DEVICE)

# The canonical training step
optimizer.zero_grad(set_to_none=True)   # 1. clear stale grads
preds = model(x)                        # 2. forward
loss = criterion(preds, y)              # 3. loss
loss.backward()                         # 4. populate .grad
optimizer.step()                        # 5. apply update

# Gradient accumulation pattern (intentional skip of zero_grad)
ACCUM = 4
optimizer.zero_grad(set_to_none=True)
for i in range(ACCUM):
    preds = model(x)
    loss = criterion(preds, y) / ACCUM   # scale so total grad has same magnitude
    loss.backward()
optimizer.step()
optimizer.zero_grad(set_to_none=True)

## 1.4 Device placement — the CPU↔GPU thrash

**Bug:** `tensor.cpu()` or `.numpy()` or `.item()` inside the inner loop forces a sync and copies data off-device. Same for moving the model or rebuilding tensors every step.

**Fix:** push **everything** to device once. Inside the loop, only `batch.to(device, non_blocking=True)`. Use `non_blocking=True` together with `pin_memory=True` in the DataLoader for async transfer.


In [ ]:
# BAD — sync + copy every step
for x, y in loader:
    x = x.cuda()
    y = y.cuda()
    loss_value = loss.cpu().item()   # forces device sync
    metrics.append(preds.cpu().numpy())  # ditto

# GOOD
model = model.to(DEVICE)
loader = DataLoader(ds, batch_size=32, pin_memory=True, num_workers=4)
for x, y in loader:
    x = x.to(DEVICE, non_blocking=True)
    y = y.to(DEVICE, non_blocking=True)
    ...
    running_loss += loss.detach()        # stays on GPU, accumulate cheaply
epoch_loss = (running_loss / num_batches).item()  # sync ONCE at end

On Ampere+ (A100, H100, RTX 30/40-series), `bfloat16` has the same dynamic range as `fp32` so you can drop the `GradScaler` entirely.
- Less code, fewer footguns.
- `fp16` is the right call if you need to support older GPUs; `bf16` is the right call if you control the hardware.
- At GM on embodied AI workloads we'd typically know our target hardware.

## 1.5 Vectorization — kill the Python loop

**Bug:** Python-level loops doing math the GPU could batch. Each `.item()`, `.cpu()`, or scalar op forces a launch + sync.

**Fix:** replace with `matmul`, `einsum`, broadcasting, or `torch.vmap` for harder cases.


In [ ]:
# Pairwise dot products between rows of A and rows of B
A = torch.randn(64, 128, device=DEVICE)
B = torch.randn(64, 128, device=DEVICE)

# BAD — Python loop
# out_bad = torch.empty(64, device=DEVICE)
# for i in range(64):
#     out_bad[i] = torch.dot(A[i], B[i])

# GOOD — batched
out_good = (A * B).sum(dim=-1)                # element-wise then reduce
# or
out_good = torch.einsum("ij,ij->i", A, B)     # einsum is the swiss army knife
# or full pairwise matrix
pairwise = A @ B.T                            # (64, 64)

print(out_good.shape, pairwise.shape)

## 1.6 Bonus fixers worth memorizing

If the prompt is "improve this training code," these are easy wins that show seniority:
- **Mixed precision** (`torch.amp`) — ~2x throughput on Ampere+ with no accuracy loss for most models
- **Gradient clipping** — prevents the occasional exploding gradient from poisoning a checkpoint
- **`set_to_none=True`** in `zero_grad` (already the default in 2.x but call it out)
- **`channels_last` memory format** for conv-heavy vision models
- **`torch.compile(model)`** in PyTorch 2.x — usually free speedup


In [ ]:
# Mixed precision + gradient clipping — modern training step
from torch.amp import autocast

model = nn.Linear(10, 1).to(DEVICE)
model = torch.compile(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
MAX_GRAD_NORM = 1.0

x = torch.randn(32, 10, device=DEVICE)
y = torch.randn(32, 1, device=DEVICE)

optimizer.zero_grad(set_to_none=True)
with autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=(DEVICE.type == "cuda")):
    preds = model(x)
    loss = criterion(preds, y)

loss.backward()
optimizer.step()

autocast wraps math you want in low precision; accumulation, scaling, and bookkeeping stay in fp32

Important: You must unscale the gradients *before* you clip them.
- if you clip them while they are still multiplied by the scaler's huge factor, your `MAX_GRAD_NORM` threshold will be meaningless, and you'll clip almost every gradient

`scaler.step(optimizer)` replaces `optimizer.step()` because its checking for gradients resulted in `Inf` or `NaN` (which can happen in mixed precision)
- if the gradients are safe, it applies them
- if they contain `NaN`s it skips the step to protect the model

`scaler.update()` - this adjusts the scaling factor for the next batch
- if the last step had NaNs, it shrinks the multiplier
- if it's been safe for awhile, it increases the multiplier

AMP issues would show up as `NaN` or `fp16-overflow`, not a smooth plateau.

Under `fp16` AMP, gradients come out of `.backward()` multiplied by the loss scale factor (typically 2^16 or so, dynamically adjusted).
- If you call `clip_grad_norm_(max_norm=1.0)` on those scaled gradients, your effective clip threshold in real-gradient space is `1.0 / 2^16 ≈ 1.5e-5`.
- You haven't clipped the gradients — you've nuked them.
- The model trains as if the LR were a millionth of what you set.
- `scaler.unscale_` divides gradients back to true magnitudes first, so `max_norm=1.0` actually means what it says.
- `NaN` isn't really the failure mode here; silent under-training is.

---
# Focus Area 2 — The Builder

Memorize each of these blocks. If they say "implement a small CNN" or "write attention," you should be typing within 15 seconds.


## 2.1 The `nn.Module` skeleton

The canonical structure. Drill this until you can write it without thinking.


In [ ]:
class MyModule(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()                          # ALWAYS first line
        self.fc = nn.Linear(in_dim, out_dim)        # register submodules here

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

# Verify with a shape check — do this for every module you build
m = MyModule(16, 4)
out = m(torch.randn(2, 16))
print(out.shape)   # torch.Size([2, 4])

## 2.2 Conv block: Conv → BN → ReLU (CBR)

The fundamental vision building block. Variants swap `BN` for `GroupNorm`/`LayerNorm` and `ReLU` for `GELU`/`SiLU`.

**Shape math to memorize for `Conv2d`:**

`H_out = (H_in + 2*padding - kernel_size) / stride + 1`

So `kernel=3, padding=1, stride=1` preserves spatial size. `stride=2` halves it.


In [ ]:
class ConvBlock(nn.Module):
    """Conv -> BN -> ReLU, the workhorse of vision backbones."""
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int = 3,
                 stride: int = 1, padding: int = 1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size,
                              stride=stride, padding=padding, bias=False)  # bias=False because BN follows
        self.bn   = nn.BatchNorm2d(out_ch)
        self.act  = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class TinyCNN(nn.Module):
    """Minimal classifier — stack of CBR + MaxPool + GAP + Linear."""
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.stem  = ConvBlock(3, 32)
        self.block = ConvBlock(32, 64, stride=2)
        self.pool  = nn.AdaptiveAvgPool2d(1)                # global average pool
        self.head  = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.block(x)
        x = self.pool(x).flatten(1)                         # (B, 64, 1, 1) -> (B, 64)
        return self.head(x)

# Shape check
print(TinyCNN()(torch.randn(2, 3, 64, 64)).shape)   # torch.Size([2, 10])

Whenever you halve the spatial resolution (using `stride=2`), you should double the number of channels
- as the image gets physically smaller as it passes through the network, you increase the channel depth to compensate and store more complex, abstract information (like "this is a dog's ear" instead of just "this is a curve")

Rule to Memorize: You *write* the `forward` method to define the math, but you *never call it directly*
- always treat the model object itself as a function: `model(x)` never `model.forward(x)`

## 2.3 Residual block

If they ask for "ResNet-style," this is what they mean. The skip connection lets gradients flow even when the conv stack is deep.

**Trap:** if `in_ch != out_ch` or `stride != 1`, the identity needs a `1x1 conv` projection to match shapes.


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)

        # Project identity if dimensions don't match
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = out + identity                                 # residual add BEFORE final activation
        return F.relu(out, inplace=True)

print(ResidualBlock(32, 64, stride=2)(torch.randn(2, 32, 32, 32)).shape)

Notice that `conv1` takes the `stride` argument but `conv2` forces `stride=1`
- if we need to shrink the spatial dimensions of the image (e.g. 32x32 to 16x16), we do it all at once in the first layer
- the second layer just processes the smaller feature map

Why `bias=False`
- BN subtracts the mean of the batch
- if your convolution learned a bias (a constant addition), the very next `BatchNorm2d` layer would immediately subtract it right back out
- setting `bias=False` saves memory and unnecessary parameters

The whole point of a residual block is the math operation $F(x) + x$, but what happens if $F(x)$ (the output of the main convolutions) is shaped (64, 16, 16) and $x$ (the original input) is shaped (32, 32, 32)
- Pytorch will throw a shape mismatch error as you cannot add them together

Fix: we use a 1x1 convolution; a 1x1 convolution doesn't look for spatial patterns (edges, shapes) because it's only 1 pixel wide, its only job is to act like channel mixer
- it projects the 32 input channels into 64 output channels
- if `stride=2`, it also drops every other pixel, perfectly matching the 16x16 spatial shape of the main pathway

If the shapes already match naturally, `nn.Identity()` does nothing
- it just passes the raw input forward

Notice that there is no ReLU immediately after `bn2`; the addition `out = out + identity` happens *before* the final ReLU activation, why?
- because we want the network to be able to learn the exact identity function it if needs to
- if we applied ReLU before the addition, we would force the main pathway to only output positive numbers, destroying its ability to represent negative residuals (which are necessary to cancel out the identity path if a layer isn't needed)




## 2.4 MLP block with dropout

Standard for the feed-forward sub-layer in Transformers (typically `4x` expansion).
- you are looking at the Position-wise Feed-Forward Network (FNN) which makes up the second half of every single Transformer block
- the exact same linear weights are applied independently to every single token in the sequence

If self-attention is how tokens "talk" to each other, this MLP is how each individual token "processes" what it just learned

In [ ]:
class MLP(nn.Module):
    def __init__(self, dim: int, hidden: int | None = None, dropout: float = 0.1):
        super().__init__()
        hidden = hidden or 4 * dim                          # transformer convention
        self.net = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

print(MLP(128)(torch.randn(2, 10, 128)).shape)   # (B, seq, dim) preserved

`dim` = the embedding dimension of the model (also called `d_model`)

`hidden` = the size of the inner layer

If no hidden dimension is provided, it defaults to exactly 4 times the input dimension
- the FFN projects the representation into a much higher-dimensional space (usually 4x) and then projects it back down
- the expansion creates a massive parameter space where the model stores its actual "knowledge" and learned features

Think of Attention as the *routing* mechanism, and this FFN as the *memory* or *compute* mechanism

For modern LLMs, they switched from `ReLU` to `GELU` (Gaussion Error Linear Unit) or `SwiGLU`
- unlike `ReLU` (which sharply cuts off negative values at exactly zero), `GELU` has a smooth, curved drop-off for negative values
- this smoothness helps gradients flow better during backpropagation and prevents the "dead neuron" problem

## 2.5 Scaled dot-product attention

The single most common interview ask. Memorize this exact function.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\tfrac{QK^\top}{\sqrt{d_k}}\right) V$$

The $\sqrt{d_k}$ scaling keeps softmax inputs from saturating as dimension grows.


In [ ]:
def scaled_dot_product_attention(
    q: torch.Tensor,                 # (B, ..., Lq, d_k)
    k: torch.Tensor,                 # (B, ..., Lk, d_k)
    v: torch.Tensor,                 # (B, ..., Lk, d_v)
    mask: torch.Tensor | None = None # (B, ..., Lq, Lk) — True/1 = keep, False/0 = mask
) -> torch.Tensor:
    d_k = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)        # (B, ..., Lq, Lk)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    attn = F.softmax(scores, dim=-1)
    return attn @ v                                          # (B, ..., Lq, d_v)

# In PyTorch >= 2.0 the production-ready version is:
# F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=False)
# It uses FlashAttention when available. Mention this in the interview.

B, L, D = 2, 8, 32
q = k = v = torch.randn(B, L, D)
print(scaled_dot_product_attention(q, k, v).shape)   # (2, 8, 32)

## 2.6 Multi-Head Attention

The standard pattern: project to `Q, K, V`, reshape to `(B, n_heads, L, d_head)`, run scaled DPA, concatenate, project out.


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, dim: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"
        self.n_heads = n_heads
        self.d_head  = dim // n_heads

        # One fused projection is faster than three separate ones
        self.qkv  = nn.Linear(dim, 3 * dim, bias=False)
        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, L, D = x.shape
        qkv = self.qkv(x)                                    # (B, L, 3D)
        qkv = qkv.reshape(B, L, 3, self.n_heads, self.d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)                     # (3, B, n_heads, L, d_head)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # Use the fused kernel in production; the manual form is here for clarity
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        out = out.transpose(1, 2).contiguous().reshape(B, L, D)   # (B, L, D)
        return self.drop(self.proj(out))

print(MultiHeadAttention(64, 8)(torch.randn(2, 16, 64)).shape)

A single attention head can only focus on one representation subspace at a time
- by splitting the embedding dimension into multiple heads, the model can look at different types of relationships simultaneously

We must be able to divide the embedding dimension `dim` equally among the heads

In the `self.qkv  = nn.Linear(dim, 3 * dim, bias=False)` line, doing 3 seperate `nn.Linear` calls means launching 3 seperate CUDA kernels on the GPU and reading the input tensor $x$ from memory 3 times
- fusing it into one `3*dim` matrix multiplication reduces kernel launch overhead and maximizes memory bandwidth
- we drop the `bias` here because it is mathematically redundant if this layer is immediately preceded or followed by `LayerNorm` (which has its own bias)

**Tensor Math**:
The goal is to manipulate the tensor shapes so that the batch `B` and the heads `n_heads` act as independent, parallel batch dimensions during the attention calculation

```
# Fusing and Reshaping
qkv = self.qkv(x)     # (B, L, 3D)
qkv = qkv.reshape(B, L, 3, self.n_heads, self.d_head)
```

Here we project our input sequence into a massive tensor.
- then, we use `reshape` to break that `3D` dimension down into its components: the 3 distinct matrics (`q, k, v`), the number of heads `n_heads`, and the dimension per head `d_head`

```
# The Permute (The "Aha!" Moment)
# for index purposes: qkv.reshape(B, L, 3, self.n_heads, self.d_head)
qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, n_heads, L, d_head)
```

This single line is the most critical routing step in the entire transformer architecture; `permute` reorders the axes. Lets trace the indices:
- `[2]` moves to `0` -> 3, we pull the `q, k, v` dimension to the very front
- `[0]` moves to `1` -> `B`, the batch size
- `[3]` moves to `2` -> `n_heads`, we move the heads right next to the batch dimension
- `[1]` moves to `3` -> `L`, the sequence length
- `[4]` stays at `4` -> `d_head`, the feature dimension of each head

By putting `B` and `n_heads` at the very front, Pytorch treats them as a single, combined batch dimension when calculating the dot product later

```
# Unpacking and Attention
q, k, v = qkv[0], qkv[1], qkv[2]
out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
```

Because of our `permute`, `q, k, v` now have the shape `(B, n_heads, L, d_head)`
- pytorchs native function handles the heavy lifting in parallel across all heads
- the output `out` retains this same shape

```
# Reassembling the Heads
out = out.transpose(1, 2).contiguous.reshape(B, L, D)
```

When you use `transpose` (swapping `n_heads` and `L` back to their original order, Pytorch does not actually move the data in the GPU memory; it just creates a "view" with updated metadata
- however, `reshape` usually requires the underlying memory to be laid out sequentially (contiguously)
- calling `contiguous()` forces pytorch to physically rewrite the tensor in memory so that `reshape` can safely flatten the `n_heads` and `d_head` dimension back into a single $D$ dimension

```
# The Final Mix
return self.drop(self.proj(out))
```

Finally, we pass the concatenated heads through an output projection `self.proj`
- this allows the different heads to communicate and mix their learned features together before passing the output to the next layer of the Transformer

## 2.7 ViT patch embedding

Turn an image into a sequence of tokens. The clever trick: a single strided conv does both patching AND linear projection.

To feed a 2D image into a Transformer, you have to cut it into non-overlapping patches (like a grid) and treat each patch like a "word" (or token)
- if your patch is 16x16 pixels with 3 color channels, flattening it gives you a vector of length 16x16x3=768
- you then need to project this vector into your model's embedding dimension

Instead of cropping, flattening, and passing through a `nn.Linear` layer, a single convolutional layer with `kernel_size = stride = patch_size`achieves the exact same thing
- because the stride equals the kernel size, the convolution window "jumps" exactly one patch width at a time
- it never overlaps, effectively treating each patch as an independent block


In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size: int = 224, patch_size: int = 16,
                 in_ch: int = 3, embed_dim: int = 768):
        super().__init__()
        assert img_size % patch_size == 0
        self.n_patches = (img_size // patch_size) ** 2

        # Conv with kernel=stride=patch_size IS the patch embedding
        self.proj = nn.Conv2d(in_ch, embed_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x):                                    # (B, 3, H, W) = (B, 3, 224, 224)
        x = self.proj(x)                                     # (B, D, H/P, W/P) = (B, 768, 14, 14)
        x = x.flatten(2).transpose(1, 2)                     # (B, n_patches, D)
        return x

print(PatchEmbed()(torch.randn(2, 3, 224, 224)).shape)   # (2, 196, 768)

In ViT, the image dimensions must be perfectly divisible by the patch size, otherwise you'll have leftover pixels at the edges that the convolution will drop or crash on.

Transformers scaled $O(N^2)$ with sequence length, so knowing exactly how many tokens you are generating is critical for understanding memory constraints.

`n_patches` calculates how many "tokens" (patches) the image will be broken into
- for a 224x224 image and 16x16 patches, you get a 14x14 grid, which equals 196 patches

In `x = self.proj(x)` the convolution extracted a 768-dimensional feature vector for each of the 14x14 patches
- at this point, it is currently a 2D spatial grid of embeddings

The `.flatten(2)` method tells pytorch to collapse all dimensions starting from index 2
- it merges the height `14` and the width `14` into single sequence length dimension `196`

The `.tranpose(1, 2)` swaps the dimension 1 `embed_dim` and dimension 2 `sequence_length` to match the standard NLP format
- `(B, 786, 196)` -> `(B, 196, 768)`
- pytorch's native transformer layers expect the input in the format `(B, L, D)` or `(Batch, Sequence, Features)`

## 2.8 The Transformer Block

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim=96, num_heads=4, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        hidden = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, embed_dim),
        )

    def forward(self, x):
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, need_weights=False)
        x = x + attn_out    # residual on RAW x
        x = x + self.mlp(self.norm2(x))   # norm is a branch, not the highway
        return x

## 2.9 U-Net building blocks

Relevant for any dense-prediction task in embodied AI (depth, segmentation, BEV occupancy). The trick is the skip connections from encoder to decoder at matching resolutions.

For dense-prediction tasks in embodied AI (like mapping an image to a depth map or a Bird's-Eye-View occupancy grid), you can't just output a single label like "cat"
- you have to output a prediction for every single pixel

To do this, the network needs to understand "what" is in the image (high-level semantics, which requires downsampling) and "where" it is (high-resolution spatial details, which requires upsampling)

U-Net bridges this gap by using an encoder-decoder structure tied together with skip connections.


In [ ]:
class ConvBlock(nn.Module):
    """Conv -> BN -> ReLU, the workhorse of vision backbones."""
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int = 3,
                 stride: int = 1, padding: int = 1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size,
                              stride=stride, padding=padding, bias=False)  # bias=False because BN follows
        self.bn   = nn.BatchNorm2d(out_ch)
        self.act  = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

class DoubleConv(nn.Module):
    """Two CBR blocks — the U-Net unit."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            ConvBlock(in_ch, out_ch),
            ConvBlock(out_ch, out_ch),
        )

    def forward(self, x):
      return self.block(x)


class Down(nn.Module):
    """Downsample then DoubleConv."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x):
      return self.conv(self.pool(x))


class Up(nn.Module):
    """Upsample, concat skip, DoubleConv."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        # ConvTranspose halves channels and doubles spatial size
        self.up   = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_ch, out_ch)               # in_ch = up + skip

    def forward(self, x, skip):
        x = self.up(x)
        # Handle off-by-one mismatches from odd input sizes
        dy, dx = skip.size(-2) - x.size(-2), skip.size(-1) - x.size(-1)
        x = F.pad(x, [dx // 2, dx - dx // 2, dy // 2, dy - dy // 2])
        x = torch.cat([skip, x], dim=1)                     # concat along channel dim
        return self.conv(x)

`DoubleConv` just chains two `ConvBlock`s together
- in U-Net, we typically do two convolutions at every spatial resolution before scaling up or down

The `Down` module represents the "contracting" path
- `MaxPool2d(2)` cuts the spatial dimensions (height and width) in half
- the `DoubleConv` then processes this smaller map, usually doubling the number of channels

By shrinking the spatial size, the network increases its receptive field (each pixel in the deeper layers "sees" a larger portion of the original image)
- this is how the model learns the high-level "what" (e.g. "this blob of pixels is a car")

`ConvTranspose2d` (often called a deconvolution) does the opposite of max-pooling
- with a stride of 2, it doubles the spatial dimensions and halves the channels; it begins reconstructing the image

**The Padding Logic**

If your input image is 128x128, downsampling gives you 64x64, and upsampling gives 128x128, but what if your input was 127x127?
- max-pooling rounds down to 63x63 -> upsampling doubles it to 126x126

You now have a size mismatch: your upsampled feature map `x` is 126x126, but your encoder feature map `skip` is 127x127

`F.pad` dynamically calculates the difference (`dy`, `dx`) and pads the upsampled map `x` so it perfectly matches the `skip` tensor
- pytorch padding format starts from the last dimension and moves backward: `[left, right, top, bottom]`

**The Skip Connection** (`torch.cat`): The model concatenates the high-resolution features from the encoder (`skip`) with the upsampled high-level features (`x`) along the channel dimension (`dim=1`)
- the upsampled `x` knows what it is looking at, but it has lost fine-grained boundary details due to the downsampling earlier
- the `skip` connection passes those high-resolution edge details directly across the network
- this is how dense prediction model draws perfectly crisp boundaries for segmentation or depth masks

In [ ]:
# 1. Test DoubleConv (Encoder step)
x = torch.randn(2, 3, 128, 128)
dc = DoubleConv(3, 64)
skip = dc(x)
print("DoubleConv (skip) shape: ", skip.shape)   # Expected: (2, 64, 128, 128)

# 2. Test Down (Bottleneck step)
down = Down(64, 128)
bottleneck = down(skip)
print("Down (bottleneck) shape: ", bottleneck.shape) # Expected: (2, 128, 64, 64)

# 3. Test Up (Decoder step with skip connection)
# The Up block takes the bottleneck (128 channels), upsamples it to 64 channels,
# concatenates with the skip connection (64 channels) to make 128 channels,
# and finally applies DoubleConv to output 64 channels.
up = Up(in_ch=128, out_ch=64)
out = up(bottleneck, skip)
print("Up (decoder out) shape:  ", out.shape)      # Expected: (2, 64, 128, 128)

1. `dc = DoubleConv(3, 64)`
    - input: `(2, 3, 128, 128)` -> Batch of 2 RGB images
    - output `skip`: `(2, 64, 128, 128)` -> the spatial size stays the same, but we expand to 64 feature channels; we save this tensor to use later

2. `down = down(64, 128)`
    - input: `(2, 64, 128, 128)`
    - output `bottleneck`: `(2, 128, 64, 64)` -> max-pooling cuts the size to 64x64; `DoubleConv` increases channels to 128

3. `up = Up(128, 64)`
    - inputs: `bottleneck` `(2, 128, 64, 64)` and `skip` `(2, 64, 128, 128)`
    - Step A (Upsample): `ConvTranspose2d` transforms the bottleneck to `(2, 64, 128, 128)`
    - Step B (Concat): Concatenating the upsampled tensor and the `skip` tensor along `dim=1` creates a tensor of shape `(2, 128, 128, 128)` (since $64+64=128$
    - Step C (DoubleConv): The final convolution shrinks the channels back down to match the output size, resulting in `(2, 64, 128, 128)`

---
# Focus Area 3 — The Mechanic

Functional pieces around the model: data, debugging, hypothesis tests. This is where the "create hypotheses and test them" criterion gets graded.


## 3.1 `torch.utils.data.Dataset` — the three required methods

`__init__`, `__len__`, `__getitem__`. Memorize.


In [ ]:
class TensorPairDataset(Dataset):
    """Minimal supervised dataset. Replace the stub with real loading logic."""
    def __init__(self, X: torch.Tensor, y: torch.Tensor, transform=None):
        assert len(X) == len(y), "X and y must be the same length"
        self.X = X
        self.y = y
        self.transform = transform

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        x, target = self.X[idx], self.y[idx]
        if self.transform is not None:
            x = self.transform(x)
        return x, target

ds = TensorPairDataset(torch.randn(100, 3, 32, 32), torch.randn(100, 10))
print(len(ds), ds[0][0].shape, ds[0][1].shape)

If a transform was provided, you apply it to the feature `x` right before returning it in `__getitem__`
- applying transformations here (instead of in `__init__`) is vital because it saves RAM and allows for random data augmentation (like a random crop that looks slightly different every time `idx` is called)

`ds[0]` calls your `__getitem__(0)`, returning the tuple `(x, target)`
- the first element `[0]` is the image `x`
- because you sliced the 100-item tensor, you are left with one image of 3 channels, 32 height and 32 width

`ds[0][1].shape` outputs `torch.Size([10])`
- the second element `[1]` is the `target`
- slicing the `(100, 10)` label tensor leaves a 1D tensor of size 10 (often representing 10 class probabilities

## 3.2 `DataLoader` — the knobs that matter

If they ask "why is my GPU at 30% utilization," it's almost always the loader (data starvation).

| Arg | What it does | When to set |
|---|---|---|
| `batch_size` | obvious | as large as fits |
| `shuffle=True` | shuffles every epoch | always for train, never for val |
| `num_workers` | CPU processes for parallel loading | start with `4 * num_gpus`, tune |
| `pin_memory=True` | uses page-locked memory for faster H2D | always with CUDA |
| `persistent_workers=True` | keep workers alive across epochs | always when `num_workers > 0` |
| `drop_last=True` | drops final partial batch | helps with BatchNorm + DDP |
| `collate_fn` | custom batching | needed for variable-length sequences |


In [ ]:
def pad_collate(batch):
    """Example collate for variable-length sequences."""
    xs, ys = zip(*batch)
    lengths = torch.tensor([x.size(0) for x in xs])
    xs_padded = torch.nn.utils.rnn.pad_sequence(xs, batch_first=True)
    ys = torch.stack(ys)
    return xs_padded, lengths, ys

train_loader = DataLoader(
    ds,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    drop_last=True,
    # collate_fn=pad_collate,    # only if you need it
)
print(next(iter(train_loader))[0].shape)

Your GPU is mathematical powerhouse, but it is entirely dependent on the CPU to read data from the disk, preprocess it, and send it over the PCIe bus
- if the CPU can't prepare data fast enough, the GPU sits there idle

`num_workers=4` (Crucial for GPU utilization) -> Rule of Thumb = `4*num_of_gpus`
- by default, pytorch uses the main process to fetch data (`num_workers=0`); if data loading takes time, the GPU waits
- setting this to > 0 spins up background subprocesses that pre-fetch and process the next batches while the GPU is busy training on the current one

`pin_memory=True` (Crucial for GPU Utilization)
- normally, host (CPU) memory is "pageable", meaning the OS can swap it to the hard drive
- GPUs cannot safely read from pageable memory, so Pytorch has to secretly copy the data to a temporary "pinned" (page-locked) memory block before sending it to the GPU
- setting `pin_memory=True` allocates the tensors in pinned memory from the start, enabling much faster, direct memory transfers to the GPU

`persistent_workers=True`
- when an epoch ends, Pytorch normally shuts down the worker processes and restarts them for the next epoch
- this creates a noticeable delay (a spike in GPU idle time) between epochs
- this flag keeps the workers alive, eliminating that overhead

`next(iter(train_loader))[0].shape`
- this turns the `train_loader` into an iterator, grabs the very first batch, selects the first element returned by the collate function (which is `xs_padded`) and prints its shape
- its a standard sanity check to ensure your data pipeline isn't broken before you start training

**Custom Collate Function**

By default, the DataLoader assumes every item in your batch is the exact same shape, and it just stacks them into a new dimension using `torch.stack`
- but what if you are doing NLP (sentences have different word counts) or Audio processing (clips have different durations)
- the default loader will crash because you can't stack a tensor of size `(10, 512)` with one of `(15, 512)`

That's where `pad_collate` comes in.
- the `collate_fn` intercepts a python list of samples from your dataset and tells Pytorch exactly how to merge them into a single batch

`xs, ys = zip(*batch)`
- batch is a list of tuples: `[(x1, y1), (x2, y2), ...]`
- `zip(*batch)` unzips the list of tuples into two lists: `xs = (x1, x2, ...), ys = (y1, y2, ...)`

`lengths = torch.tensor([x.size(0) for x in xs])`
- we record the original length of each sequence before we alter them
- this is often needed later for models like LSTMs (using `pack_padded_sequence`)

`xs_padded = torch.nn.utils.rnn.pad_sequence(xs, batch_first=True)`
- this is the magic; it finds the longest tensor in `xs` and pads the rest with zeros (by default) so they all match that maximum length
- `batch_first=True` makes the output shape `(batch_size, max_seq_len, features)`

`ys = torch.stack(ys)`
- the labels (ys) are usually just single integers or fixed-size vectors, so we can stack them normally

`return xs_padded, lengths, ys`
- we return the batch of padded inputs, their real lengths, and the labels

## 3.3 Tensor debugging — the diagnostic toolkit

When a tensor "doesn't look right," these four attributes are your first move. **Always print all four** — the bug is usually in the one you didn't check.


In [ ]:
x = torch.randn(2, 3, 4, device=DEVICE)

# The fab four
print("shape :", x.shape)        # most common mismatch source
print("dtype :", x.dtype)        # float32 vs float16 vs long — silent bug if wrong
print("device:", x.device)       # cpu vs cuda mismatch raises a runtime error
print("contig:", x.is_contiguous())  # matters for .view() vs .reshape()

# Hunting NaN / Inf — happens with exploding grads, log(0), divide-by-zero
print("nan?", torch.isnan(x).any().item())
print("inf?", torch.isinf(x).any().item())
print("range:", x.min().item(), x.max().item())

# Quick stats — useful for activations and gradients
print("mean/std:", x.mean().item(), x.std().item())

`x.is_contiguous()` is the sneakiest error of the 4
- its a boolean check of whether the tensor's values are stored in a continuous block of memory in the same order as they are indexed

Operations like `.transpose()` or `.permute()` don't actually move numbers around in memory; they just change the "stride" (the indexing math)
- this makes the tensor "non-contiguous"
- if you then try use `.view()` to flatten that tensor, pytorch will crash
- you have to call `.contiguous()` first, or use `.reshape()` (which handles the memory copying under the hood)

`.any().item()`
- `torch.isnan(x)` returns a tensor of booleans the same shape as `x`
- adding `.any()` collapses it into a single boolean (`True` if *any* value is `NaN`
- adding `.item()` safely pulls that single value out of the pytorch graph and turns it into a standard python boolean

If even a single `NaN` appears in your network, it will poison the gradients, and suddenly all your model weights will become `NaN`s

## 3.4 Gradient inspection — is this layer learning?

After `.backward()`, every leaf tensor with `requires_grad=True` has a `.grad`. If a layer's `.grad` is `None`, all zeros, or all NaN, that layer is not training.


In [ ]:
model = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 1),
).to(DEVICE)

x = torch.randn(8, 10, device=DEVICE)
y = torch.randn(8, 1, device=DEVICE)

loss = F.mse_loss(model(x), y)
loss.backward()

# Per-parameter gradient health
for name, p in model.named_parameters():
    if p.grad is None:
        print(f"{name}: NO GRAD — layer is frozen or detached")
    else:
        g = p.grad
        print(f"{name:20s} norm={g.norm().item():.4f} "
              f"nan={torch.isnan(g).any().item()} "
              f"max={g.abs().max().item():.4f}")

# Aggregate health check — useful in the training loop
total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float("inf"))
print(f"total grad norm: {total_norm:.4f}")

When you pass data through a Pytorch model, it doesn't just calculate numbers; it builds a map of every mathematical operation performed
- if you call `loss.backward()`, pytorch traverses this map in reverse, applying the chain rule to calculate how much each parameter contributed to the final loss
- it takes these computed derivatives and stores them in the `.grad` attribute of every "leaf tensor" (your model's weights & biases) that has `requires_grad=True`

`for name, p in model.named_parameters():`
- iterates through every parameter in the model, names will be strings like "0.weight" or "2.bias" and `p` is the actual tensor

A `.grad` of `None` means this parameter was completely disconnected from the computational graph that produced the loss; the most common answers are:
- you accidentally froze the layer (`requires_grad=False`)
- you bypassed the layer during the forward pass (e.g. an `if` statement skipped it)
- you used a non-differentiable operation like `torch.argmax` that broke the gradient flow

`norm`: the $L_2$ norm (Euclidean length) of the gradient tensor
- if this number is tiny (e.g. $10^{-5}$), your network suffers from vanishing gradients
- if its massive, you have exploding gradients

If a gradient becomes `NaN`, the moment you call `optimizer.step()`, those `NaN`s will infect your model weights, permanently ruining your model
- `NaN`s usually occur due to division by zero, taking the `log()` of zero or a negative number, or extreme exploding gradients that overflow to infinity and then evaluate to `NaN`

`clip_grad_norm_` is designed to prevent exploding gradients by scaling down the gradients if their total combined norm excees `max_norm`
- by setting `max_norm=float("inf")`, you are telling pytorch *not* to clip anything, but to still calculate and return the $L_2$ norm across the entire model
- logging `total_norm` in your tracker, if you see a sudden spike in `total_norm` during training, it usually precedes a loss explosion

## 3.5 The canonical training loop — write this from memory

If they say "implement a training loop," this is the answer. Hits AMP, clipping, LR scheduling, and proper eval, all without ceremony.


In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, criterion, device,
                    max_grad_norm: float = 1.0):
    model.train()
    running_loss = 0.0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=device.type, dtype=torch.bfloat16, enabled=(device.type == "cuda")):
            preds = model(x)
            loss  = criterion(preds, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        scheduler.step()                                    # if it's a per-step scheduler

        running_loss += loss.detach()
    return (running_loss / len(loader)).item()


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = torch.zeros(1, device=device)
    total_correct = torch.zeros(1, device=device)
    total = 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        with autocast(device_type=device.type, dtype=torch.bfloat16, enabled=(device.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)
        total_loss += loss.detach() * x.size(0)
        total_correct += (logits.argmax(dim=-1) == y).sum()
        total += x.size(0)
    return (total_loss / total).item(), (total_correct / total).item()

## 3.6 Checkpointing — save the state_dict, not the model

**Bug:** `torch.save(model, path)` pickles the whole class. Brittle across refactors.

**Fix:** save `state_dict()` dicts. Save model + optimizer + scheduler + epoch for resumable training.


In [ ]:
# Save
ckpt = {
    "epoch": 0,
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "scheduler": scheduler.state_dict(),
    "scaler": scaler.state_dict(),
}
torch.save(ckpt, "checkpoint.pt")

# Load
ckpt = torch.load("checkpoint.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"])
optimizer.load_state_dict(ckpt["optimizer"])
start_epoch = ckpt["epoch"] + 1

When you use `torch.save(model, path)`, pytorch uses Python's built-in pickle module to serialize the entire object
- the fatal flaw is that pickle does not save the model's class definition (the actual code); it only saves a path to the file containing the class
- if you change the name of the file, move the class to a different directory, or even slightly alter the class structure (like adding a new layer during a refactor), your saved model will instantly break with an `AttributeError` or `ModuleNotFoundError` when you try to load it

A `state_dict` is a Python dictionary that maps each layer in your model to its parameter tensors (weights and biases)
- it contains zero code and zero structural logic, just the raw math
- because its just data, it is entirely immune to code refactors

To do resumable training (checkpointing), you need to save the state of everything involved in the training loop, not just the model.

If you don't save the `optimizer` state, resuming training will effectively "cold start" your optimizer, causing a massive, destructive spike in your loss curve.

LR schedulers change the Learning Rate based on the epoch number
- if you don't save this, your LR will reset to its initial (usually much higher) value upon resuming

`scaler` is used for Automatic Mixed Precision (AMP)
- the scaler adjusts loss scaling factors dynamically to prevent underflow when using `torch.float16`
- it needs to remember its current scale factor

`map_location=DEVICE` forces pytorch to remap the tensors to whatever hardware is currently available

## 3.7 Reproducibility — the seed boilerplate

Always set this at the top. The interviewer might ask "how do you make this deterministic?" — having a one-liner ready is a tell of seniority.


In [ ]:
def set_seed(seed: int = 42, deterministic: bool = False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        # Slower but bit-exact reproducibility
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True)
    else:
        # Default: best perf, results within float tolerance
        torch.backends.cudnn.benchmark = True

set_seed(42)

## 3.8 Hypothesis test: overfit a single batch

**The single best diagnostic in ML.** If your model can't drive loss to ~0 on a single fixed batch, the model is broken. If it can, the data pipeline or LR is the culprit.

This pattern explicitly demonstrates the "create hypotheses and test them" criterion. If the interviewer asks "how do you know if your model is set up correctly?", **this is the answer**.


In [ ]:
def overfit_one_batch(model, criterion, optimizer, x, y, max_steps: int = 200,
                       target_loss: float = 1e-3):
    """Returns step at which loss < target, or max_steps if never."""
    model.train()
    for step in range(max_steps):
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        # early termination check
        if loss.item() < target_loss:
            return step, loss.item()
    return max_steps, loss.item()

# Example
model = nn.Linear(10, 1).to(DEVICE)
opt   = torch.optim.AdamW(model.parameters(), lr=1e-2)
x = torch.randn(8, 10, device=DEVICE)
y = torch.randn(8, 1, device=DEVICE)
step, final = overfit_one_batch(model, nn.MSELoss(), opt, x, y)
print(f"converged at step {step} with loss {final:.6f}")

In ML, bugs can live in the data, the architecture, the loss function, or the optimizer
- by fixing the data to a single, tiny batch and running the training loop, you eliminate the dataset as a variable

You are testing a specific hypothesis: "Is this architecture physically capable of mapping these inputs to these outputs?"
- if it passes (loss goes to ~0):
    - your model's gradient flow, architecture, and loss calculation are fundamentally sound
    - if your actual training run still fails, you now know the culprit is elsewhere (e.g. LR, bad data shuffling, dirty labels, or lack of capacity to generalize)
- if it fails (loss stays high or diverges)
    - you have a systematic bug
    - you might have a disconnected computational graph (gradients aren't flowing), a mismatched loss function (e.g. CrossEntropy on normalized targets), or an aggressive bottleneck in the network (like a severe vanishing gradient)

Because `loss` is a pytorch tensor attached to the computational graph, calling `.item()` extracts it as a standard Python float
- this prevents a massive memory leak that would occur if you stored the raw tensor history

## 3.9 - The "won't overfit on a single batch" diagnostic ladder

Before writing any diagnostic code, classify the failure. Each symptom narrows the hypothesis space.

| Symptom | First-order hypotheses (in order of likelihood) |
|---|---|
| Loss → NaN | fp16 overflow / missing GradScaler, exploding gradients, log(0) or 1/0 in loss, bad mask producing all-`-inf` softmax row |
| Loss flat from step 0 | Frozen params (`requires_grad=False`), disconnected graph (`.detach()` somewhere), zero LR, dead activations everywhere |
| Loss decreases then plateaus above zero on single-batch overfit | **Vanishing gradients** (Post-LN, deep nets w/o residuals), LR too low, capacity bottleneck, target normalization off |
| Loss decreases then oscillates / spikes | LR too high, missing grad clipping, batchnorm with tiny batch |
| Train loss good, val loss bad immediately | Train/val data drift, eval-time `model.eval()` missing, leaking test info |
| Train loss good, val loss diverges over time | Standard overfitting — expected, not a bug |
| OOM partway through epoch | Loss accumulation as tensor (not `.item()`), no `with torch.no_grad()` in val, growing list of unfreed activations |


### 3.9.1 — Diagnostic Ladder (Single-Batch Overfit Plateau)



This is the canonical "model won't overfit a single batch" debug. Walk it in order — don't skip steps.

1. **Gradient flow** → run 3.9.4 Look for orders-of-magnitude differences between early and late layer norms.
2. **Weight movement** → run 3.9.5 Confirm params actually update; a parameter with `delta_norm ≈ 0` is effectively frozen.
3. **Activation flow** → run 3.9.6 Look for std collapse (→0) or explosion (>10) anywhere in the network.
4. **Prediction sanity** → run 3.9.7 Is the model collapsing to the mean of the targets?
5. **LR sweep** → run 3.9.8 If lowering LR drops the plateau, you had instability; if raising LR helps, you were stuck in a flat region.
6. **Bisect the architecture** → run 3.9.9 Strip to `Linear → head` and add components back one at a time.

### 3.9.2 — Diagnostic Ladder (NaN Loss)



Different ladder for the NaN case. Run in order.

1. **Anomaly detection** → wrap forward in `torch.autograd.set_detect_anomaly(True)`. Slow but tells you the exact op that produced NaN.
2. **Check the input batch** → `torch.isnan(obs).any()`, `torch.isinf(obs).any()`. NaN in → NaN out.
3. **Check the loss inputs** → log `predictions.min/max/mean`, `target.min/max/mean` right before the loss call.
4. **Look for log(0), div(0), sqrt(negative)** — most NaN bugs are in custom loss code.
5. **Mask-induced NaN** → if using attention with masking, check whether any query row has *all* keys masked. All `-inf` row → softmax produces `0/0` → NaN. Fix with `torch.finfo(scores.dtype).min` instead of `-inf`.
6. **AMP** → if using fp16 autocast, confirm `GradScaler` is wired correctly: `scaler.scale(loss).backward()`, `scaler.unscale_(optimizer)` before clipping, `scaler.step(optimizer)`, `scaler.update()`. Try bf16 to sidestep range issues.

### 3.9.3 — Healthy Signatures (Reference Values)



When you run a diagnostic, you need to know what "fine" looks like. These are rough but useful.

| Metric | Healthy | Warning | Broken |
|---|---|---|---|
| Per-layer grad norm | 1e-4 to 1e-1 | 1e-6 or 1e+1 | 1e-8 or NaN/Inf |
| Grad norm ratio (last layer ÷ first layer) | < 100× | 100×–10,000× | > 10,000× |
| Activation std per layer | 0.1 to 2.0 | < 0.05 or > 5 | ≈ 0 or ≈ inf |
| Weight delta norm per step | 1e-5 to 1e-3 of weight norm | < 1e-7 | exactly 0 (frozen) |
| Pred std ÷ target std on overfit | > 0.9 | 0.3–0.9 (partial collapse) | < 0.1 (full mean collapse) |

### 3.9.4 — Tool: Gradient Health

In [ ]:
# Run after loss.backward(), before optimizer.step()
def grad_health(model):
    rows = []
    for name, p in model.named_parameters():
        if p.grad is None:
            rows.append((name, "NO GRAD", None, None))
            continue
        g = p.grad
        rows.append((
            name,
            g.norm().item(),
            g.abs().max().item(),
            torch.isnan(g).any().item() or torch.isinf(g).any().item(),
        ))
    print(f"{'param':40s} {'norm':>10s} {'max':>10s} {'bad?':>6s}")
    for name, n, mx, bad in rows:
        if n == "NO GRAD":
            print(f"{name:40s} FROZEN/DETACHED")
        else:
            print(f"{name:40s} {n:10.4e} {mx:10.4e} {str(bad):>6s}")

Signatures:
- All `NO GRAD` for a sub-module → check `requires_grad`, check for `.detach()`, check the param wasn't created after the optimizer.
- Last-layer norm » first-layer norm → vanishing gradients (Post-LN, no residuals, dead activations).
- First-layer norm » last-layer norm → input dominance, possibly a giant unnormalized input feature.
- Any `bad? True` → exploding gradients, missing clip, or fp16 overflow.

The `norm()` tells you the overall energy or magnitude of the update for that specific layer
- if this is near zero, the layer isn't learning

The `abs().max()` finds the single largest gradient in the tensor

If a number gets too close to zero (underflow) or too large (overflow), its turns into an `NaN` or `Inf`
- once a `NaN` enters your network, it acts like a virus effecting everything it touches

Last layer >> First Layer = Vanishing gradients

First Layer >> Last Layer = Input dominance
- if you forgot to normalize your inputs (e.g. feeding raw pixel values 0-255 instead of 0-1), those massive input values will scale up the gradients artificially at the start of your network



### 3.9.5 — Tool: Weight Movement

In [ ]:
# Snapshot before training, compare after N steps
def snapshot_weights(model):
    return {n: p.detach().clone() for n, p in model.named_parameters()}

def weight_delta(model, snapshot):
    for n, p in model.named_parameters():
        before = snapshot[n]
        delta = (p.detach() - before).norm().item()
        rel = delta / (before.norm().item() + 1e-12)
        print(f"{n:40s} Δnorm={delta:.4e} rel={rel:.4e}")

Signature: `rel ≈ 0` for any trainable param after several optimizer steps means it's effectively frozen even if `requires_grad=True`. Likely culprit is gradients of zero, not gradient blocking.



### 3.9.6 — Tool: Activation Hooks (Forward Flow)

In [ ]:
def add_activation_hooks(model, types=(nn.Linear, nn.LayerNorm)):
    stats = {}
    handles = []
    def make_hook(name):
        def hook(module, inp, out):
            t = out if isinstance(out, torch.Tensor) else out[0]
            stats[name] = {
                "norm": t.norm().item(),
                "std":  t.std().item(),
                "max":  t.abs().max().item(),
                "nan":  torch.isnan(t).any().item(),
            }
        return hook
    for name, module in model.named_modules():
        if isinstance(module, types):
            handles.append(module.register_forward_hook(make_hook(name)))
    return stats, handles  # remember to remove handles when done

# Usage
stats, handles = add_activation_hooks(model)
_ = model(obs)
for name, s in stats.items():
    print(f"{name:40s} std={s['std']:.4f} max={s['max']:.4f} nan={s['nan']}")
for h in handles:
  h.remove()

Signatures:
- Activation std collapses to ≈0 deep in network → information bottleneck, gradients can't flow back through.
- Activation std balloons (>10) → unnormalized inputs to next layer, downstream LayerNorms are doing violent rescaling.
- LayerNorm outputs uniformly ≈1.0 std but the sub-layer that fed into it has tiny std → sub-layer is contributing almost nothing to the residual stream (Post-LN failure mode).



### 3.9.7 — Tool: Prediction Sanity

In [ ]:
@torch.no_grad()
def prediction_sanity(model, inputs, targets):
    model.eval()
    preds = model(inputs)
    print(f"preds   mean={preds.mean().item():.4f}  std={preds.std().item():.4f}")
    print(f"target  mean={targets.mean().item():.4f}  std={targets.std().item():.4f}")
    print(f"ratio   std(pred)/std(target) = {(preds.std()/targets.std()).item():.4f}")
    print(f"per-dim residual std: {(preds - targets).std(dim=0).cpu().numpy()}")
    model.train()

Signatures:
- `std(pred)/std(target) ≈ 0` → full mean collapse. Plateau loss ≈ variance of targets.
- `std(pred)/std(target) ≈ 0.5` → partial mean collapse. Model captures some structure but can't fully fit.
- `std(pred)/std(target) ≈ 1.0` but high residual → fitting variance but wrong direction; check for sign flip or target scaling.



### 3.9.8 — Tool: LR Sweep


In [ ]:
def lr_sweep(model_fn, data, lrs=(1e-5, 1e-4, 1e-3, 1e-2), steps=200):
    """Quick LR sweep on a fresh model copy for each LR."""
    results = {}
    for lr in lrs:
        torch.manual_seed(0)
        model = model_fn()
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        losses = []
        for _ in range(steps):
            pred = model(data["x"])
            loss = F.mse_loss(pred, data["y"])
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
        results[lr] = losses[-1]
        print(f"lr={lr:.0e}  final_loss={losses[-1]:.4f}")
    return results

Signatures:
- Lower LR drops plateau → original LR was oscillating past the minimum.
- Higher LR helps → original LR was in a flat region; gradients too small to make progress.
- All LRs plateau at same loss → not an LR problem; revisit architecture.

Diagnostic sweep, not production training -- even when  you use a scheduler downstream, you still need to know what LR your model wants
- A *scheduler* (`CosineAnnealingLR`, OneCycle, linear warmup + decay) varies LR over training time, starting from one peak LR you choose
    - it assumes you've already picked a sensible peak
- An *LR Sweep* is how you find that peak in the first place
    - its the search; the scheduler is the policy



### 3.9.9 — Tool: Architecture Bisection


Mental procedure — no helper code, just a pattern:
1. Replace the whole model with just (input_proj → output_head). Does it overfit?
2. If yes, add ONE transformer block. Does it still overfit?
3. If no, that block is the culprit. Strip it: remove FFN, keep just attention. Still broken?
4. Continue until you've isolated the single sub-module that breaks training.

This is the most reliable diagnostic in your toolbox. Slowest but never lies.

### 3.9.10 — When To Use What

```
Loss is NaN ──────────────────► 3.9.2 ladder
Loss flat from start ─────────► check requires_grad, then 3.4 (look for NO GRAD)
Loss plateaus above zero ─────► 3.9.1 ladder, start with 3.4
Loss decreases too slowly ────► 3.8 (LR sweep) before anything else
Loss oscillates ──────────────► 3.8 with lower LRs; check for missing grad clip
Val loss is wrong ────────────► check model.eval(), then check eval data path
```

## 3.10 - Fixing

Vanishing Gradients:
1. Residual Connections (Skip Connections) in Transformers
  - Instead of passing the signal sequentially through every layer ($x \rightarrow F(x)$), you add the original input back to the output of the layer: $Output = F(x) + x$.
2. Drop Tanh and Sigmoid
  - Swap them out for ReLU, LeakyReLU, GELU, or Swish.
3. Adding Normalization Layers
  - Adding `BatchNorm1d/2d/3d` or `LayerNorm` helps stabilize the forward pass, which indirectly saves the backward pass.
  - Normalization forces the activations of a layer to have a mean of $0$ and a variance of $1$.
4. Kaiming (He) Initialization
  - Instead of initializing weights completely randomly, PyTorch's `torch.nn.init.kaiming_normal_` scales the initial random weights based on the size of the layer (the fan-in).

# 4 The Canonical End-to-End Pipeline

This block ties everything together: custom datasets, optimized DataLoaders, model compilation, mixed precision training, and evaluation.

In [ ]:
set_seed(42)

# 1. Data Preparation
# Dummy classification data: (B, 3, 32, 32) images, 10 classes
# Target is shape (N, ) with integer class labels for CrossEntropyLoss
X_train = torch.randn(256, 3, 32, 32)
y_train = torch.randint(0, 10, (256, ))
train_ds = TensorPairDataset(X_train, y_train)

X_val = torch.randn(64, 3, 32, 32)
y_val = torch.randint(0, 10, (64, ))
val_ds = TensorPairDataset(X_val, y_val)

# best practice DataLoader Setup
train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    num_workers = 4, # 4 * num_of_gpus
    pin_memory = True,  # speeds up transfer to GPU
    persistent_workers = True,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,  # never shuffle validation data
    num_workers = 4,
    pin_memory = True,
    persistent_workers = True
    # no drop_last -- we want to evaluate every val sample
)

# 2. Model & Optimization Setup
model = TinyCNN(num_classes=10).to(DEVICE)
model = torch.compile(model)  # the magic speedup line

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

EPOCHS = 5

# per-step scheduler - smoother LR curve than per-epoch
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS * len(train_loader)
)

# 3. Sanity Check before the real training
# if we can't drive loss to ~0 on one batch, the wiring is broken
sanity_model = TinyCNN(num_classes=10).to(DEVICE)
sanity_opt = torch.optim.AdamW(sanity_model.parameters(), lr=1e-3)
xb, yb = next(iter(train_loader))
xb, yb = xb.to(DEVICE), yb.to(DEVICE)
step, final = overfit_one_batch(sanity_model, criterion, sanity_opt, xb, yb)
print(f"[sanity] overfit one batch: step={step} loss={final}")

# 4. The master loop
print(f"\nstarting training on {DEVICE} ...")
for epoch in range(EPOCHS):

  # train for one epoch
  train_loss = train_one_epoch(
      model=model,
      loader=train_loader,
      optimizer=optimizer,
      scheduler=scheduler,
      criterion=criterion,
      device=DEVICE,
      max_grad_norm=1.0
  )

  # evaluate on the validation set
  val_loss, val_acc = evaluate(
      model=model,
      loader=val_loader,
      criterion=criterion,
      device=DEVICE
  )

  # logging
  print(f"Epoch: {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss} | "
        f"Val Loss: {val_loss} | "
        f"Val Accuracy {val_acc*100}%")

---
# Bonus — Embodied AI relevant pieces

GM's Embodied AI work spans perception, prediction, planning, and learned control. These are pieces that come up often in that domain. Even if they don't show up in the screen, they're worth having reflexive familiarity with for the loop interview that follows.


## 4.1 IoU and Non-Max Suppression

The two algorithmic primitives of object detection.

`IoU(A, B) = |A ∩ B| / |A ∪ B|` — both are areas of axis-aligned boxes.


In [ ]:
def box_iou(boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
    """Pairwise IoU. Boxes are [x1, y1, x2, y2]. Returns (N1, N2)."""
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])

    # Broadcasted pairwise intersection
    lt = torch.max(boxes1[:, None, :2], boxes2[None, :, :2])    # (N1, N2, 2)
    rb = torch.min(boxes1[:, None, 2:], boxes2[None, :, 2:])    # (N1, N2, 2)
    wh = (rb - lt).clamp(min=0)
    inter = wh[..., 0] * wh[..., 1]                             # (N1, N2)

    union = area1[:, None] + area2[None, :] - inter
    return inter / union.clamp(min=1e-6)


def nms(boxes: torch.Tensor, scores: torch.Tensor, iou_threshold: float = 0.5) -> torch.Tensor:
    """Returns indices of boxes to keep, sorted by score descending."""
    order = scores.argsort(descending=True)
    keep = []
    while order.numel() > 0:
        i = order[0].item()
        keep.append(i)
        if order.numel() == 1:
            break
        ious = box_iou(boxes[i:i+1], boxes[order[1:]]).squeeze(0)
        order = order[1:][ious <= iou_threshold]
    return torch.tensor(keep, dtype=torch.long)

# Production version: torchvision.ops.nms(boxes, scores, iou_threshold) — mention this exists

## 4.2 Focal Loss

Addresses class imbalance in dense detection. Down-weights easy examples.

$$\text{FL}(p_t) = -\alpha_t (1 - p_t)^{\gamma} \log(p_t)$$


In [ ]:
def focal_loss(
    logits: torch.Tensor,      # (N, C) raw scores
    targets: torch.Tensor,     # (N,) class indices
    alpha: float = 0.25,
    gamma: float = 2.0,
    reduction: str = "mean",
):
    ce = F.cross_entropy(logits, targets, reduction="none")     # (N,)
    p_t = torch.exp(-ce)                                        # prob of correct class
    fl = alpha * (1 - p_t) ** gamma * ce
    if   reduction == "mean":
      return fl.mean()
    elif reduction == "sum":
      return fl.sum()
    else:
      return fl

## 4.3 Smooth L1 / Huber loss

Standard for bounding box regression. Quadratic for small errors, linear for large — robust to outliers.


In [ ]:
def smooth_l1(pred: torch.Tensor, target: torch.Tensor, beta: float = 1.0):
    diff = (pred - target).abs()
    loss = torch.where(diff < beta, 0.5 * diff.pow(2) / beta, diff - 0.5 * beta)
    return loss.mean()

# Equivalent built-in: F.smooth_l1_loss(pred, target, beta=1.0)

## 4.4 Sinusoidal positional encoding

For attention-based models on sequence or trajectory data. The fixed (non-learned) form is the classic.


In [ ]:
def sinusoidal_positional_encoding(seq_len: int, dim: int) -> torch.Tensor:
    """Returns (seq_len, dim) positional encoding matrix."""
    pos = torch.arange(seq_len).unsqueeze(1).float()               # (L, 1)
    i   = torch.arange(0, dim, 2).float()                          # (dim/2,)
    div = torch.exp(i * -(math.log(10000.0) / dim))                # (dim/2,)
    pe = torch.zeros(seq_len, dim)
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

print(sinusoidal_positional_encoding(10, 32).shape)

## 4.5 Transformer encoder block

Pre-norm version (more stable to train than post-norm — this is the modern default).


In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, dim: int, n_heads: int, mlp_ratio: float = 4.0, dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = MultiHeadAttention(dim, n_heads, dropout=dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp   = MLP(dim, hidden=int(dim * mlp_ratio), dropout=dropout)

    def forward(self, x, mask=None):
        x = x + self.attn(self.norm1(x), mask=mask)         # pre-norm + residual
        x = x + self.mlp(self.norm2(x))
        return x

print(TransformerEncoderBlock(64, 8)(torch.randn(2, 16, 64)).shape)

## 4.6 Minimal REINFORCE step

If embodied AI comes up at all, expect a small RL prompt — even just "explain the policy gradient." The actual code is tiny.

$$\nabla_\theta J(\theta) = \mathbb{E}\left[\nabla_\theta \log \pi_\theta(a|s) \cdot R\right]$$


In [ ]:
class PolicyNet(nn.Module):
    """Discrete-action policy: outputs categorical logits."""
    def __init__(self, obs_dim: int, n_actions: int, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, n_actions),
        )
    def forward(self, obs): return self.net(obs)


def reinforce_step(policy, optimizer, log_probs: list[torch.Tensor],
                   rewards: list[float], gamma: float = 0.99):
    # Discounted returns
    returns, G = [], 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32)
    # Baseline / standardize for variance reduction
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)

    loss = -torch.stack([lp * R for lp, R in zip(log_probs, returns)]).sum()
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    return loss.item()

---
# Classical ML in PyTorch

If they ask for "a basic ML architecture," they probably mean a small neural net — but **linear/logistic regression in PyTorch is the sneaky variant**. It tests whether you understand that these are just `nn.Linear` + the right loss, plus they're a stepping stone to check fundamentals before scaling up.

K-means and PCA show up under the "algorithmic piece" flavor. All four below should take <2 minutes each to write.


## Linear regression

The simplest possible model: `y = Wx + b`. In PyTorch terms, that's `nn.Linear(d_in, 1)` with MSE loss.

**The senior framing:** "Linear regression is a one-layer MLP with no activation and MSE loss. Logistic regression is the same thing with cross-entropy loss. Everything else is more layers and nonlinearities."


In [ ]:
class LinearRegression(nn.Module):
    def __init__(self, in_dim: int):
        super().__init__()
        self.linear = nn.Linear(in_dim, 1)

    def forward(self, x):
        return self.linear(x)

# Train it
torch.manual_seed(0)
true_w = torch.tensor([2.0, -3.0, 0.5])
X = torch.randn(200, 3)
y = X @ true_w + 0.1 * torch.randn(200)                     # ground-truth + noise
y = y.unsqueeze(1)                                          # (N, 1) to match model output

model = LinearRegression(in_dim=3)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
criterion = nn.MSELoss()

for step in range(200):
    optimizer.zero_grad(set_to_none=True)
    loss = criterion(model(X), y)
    loss.backward()
    optimizer.step()

print("learned weights:", model.linear.weight.detach().flatten().tolist())
print("true weights   :", true_w.tolist())
print("final loss     :", loss.item())

**Closed-form alternative** — if they ask "can you do this without gradient descent?":

$$\hat{w} = (X^\top X)^{-1} X^\top y$$

This is the analytical solution to least squares. Worth knowing exists, rarely worth implementing on a screen.


In [ ]:
# Closed-form least squares (no training loop)
# Add bias column of ones to X for the intercept
X_aug = torch.cat([X, torch.ones(X.size(0), 1)], dim=1)
# Use lstsq for numerical stability over the textbook (X.T @ X)^-1 @ X.T @ y
w_hat = torch.linalg.lstsq(X_aug, y).solution
print("closed-form weights:", w_hat.flatten().tolist())

## Logistic regression

Linear regression's classification cousin. Two flavors:
- **Binary:** `nn.Linear(d, 1)` + `BCEWithLogitsLoss` (sigmoid + BCE fused, numerically stable)
- **Multiclass:** `nn.Linear(d, n_classes)` + `CrossEntropyLoss` (log-softmax + NLL fused)

**Critical:** never apply `sigmoid` or `softmax` before the loss. The fused versions are stable; the manual versions are not.


In [ ]:
# Multiclass logistic regression — literally one Linear layer
class LogisticRegression(nn.Module):
    def __init__(self, in_dim: int, n_classes: int):
        super().__init__()
        self.linear = nn.Linear(in_dim, n_classes)

    def forward(self, x):
        return self.linear(x)                               # raw logits — DO NOT softmax here

# Synthetic 3-class problem
torch.manual_seed(0)
X = torch.randn(300, 4)
y = (X[:, 0] + X[:, 1] > 0).long() + (X[:, 2] > 0.5).long()  # in {0, 1, 2}

model = LogisticRegression(in_dim=4, n_classes=3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
criterion = nn.CrossEntropyLoss()                           # expects raw logits, integer targets

for step in range(200):
    optimizer.zero_grad(set_to_none=True)
    logits = model(X)
    loss = criterion(logits, y)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    preds = model(X).argmax(dim=-1)
    acc = (preds == y).float().mean().item()
print(f"final loss: {loss.item():.4f}, train acc: {acc:.3f}")

## K-means

Classic unsupervised clustering. The two steps repeat: **assign** each point to its nearest centroid, then **update** centroids to be the mean of their assigned points.

This shows up under "algorithmic piece" — they want to see you handle tensor broadcasting and indexing cleanly.


In [ ]:
def kmeans(X: torch.Tensor, k: int, n_iters: int = 50, tol: float = 1e-4):
    """
    X: (N, D) data points
    Returns: centroids (k, D), assignments (N,)
    """
    N, D = X.shape
    # Init: pick k random points as starting centroids (k-means++ is better, this is OK for a screen)
    idx = torch.randperm(N)[:k]
    centroids = X[idx].clone()

    for it in range(n_iters):
        # Assign: pairwise squared distances, then argmin
        # X: (N, 1, D), centroids: (1, k, D) -> dists: (N, k)
        dists = ((X.unsqueeze(1) - centroids.unsqueeze(0)) ** 2).sum(dim=-1)
        assignments = dists.argmin(dim=-1)                  # (N,)

        # Update: mean of points assigned to each cluster
        new_centroids = torch.stack([
            X[assignments == j].mean(dim=0) if (assignments == j).any() else centroids[j]
            for j in range(k)
        ])

        # Convergence check
        shift = (new_centroids - centroids).norm()
        centroids = new_centroids
        if shift < tol:
            break

    return centroids, assignments

# Test on synthetic clusters
torch.manual_seed(0)
cluster_centers = torch.tensor([[0.0, 0.0], [5.0, 5.0], [0.0, 5.0]])
X = torch.cat([c + 0.5 * torch.randn(100, 2) for c in cluster_centers])

centroids, labels = kmeans(X, k=3)
print("learned centroids:")
print(centroids)

**Senior framing:** k-means is also the M-step of a Gaussian Mixture Model with fixed identity covariance and hard assignments. If they ask "what's a soft version of k-means?" → GMM with EM.

**Vectorization note:** the cluster-update loop above is `O(k)`, which is fine. A fully vectorized version uses `scatter_add_` to sum points per cluster — mention this exists if they ask about scale.


## PCA via SVD

Principal Component Analysis = find the directions of maximum variance. Implemented via SVD of the centered data matrix.

If $X \in \mathbb{R}^{N \times D}$ is centered, then $X = U \Sigma V^\top$, and the columns of $V$ are the principal directions. The top-$k$ components project as $X V_{[:, :k]}$.


In [ ]:
def pca(X: torch.Tensor, k: int):
    """
    X: (N, D) data matrix
    Returns: components (D, k), projected (N, k), explained_variance (k,)
    """
    # Center the data — PCA without centering finds direction of largest *magnitude*, not variance
    X_centered = X - X.mean(dim=0, keepdim=True)

    # SVD: X = U S V^T   where V's columns are the principal directions
    U, S, Vt = torch.linalg.svd(X_centered, full_matrices=False)
    V = Vt.T                                                # (D, D)

    components = V[:, :k]                                   # top-k directions
    projected  = X_centered @ components                    # (N, k)

    # Explained variance — singular values squared, normalized by N-1
    explained_variance = (S ** 2) / (X.size(0) - 1)
    return components, projected, explained_variance[:k]

# Test: 2D Gaussian elongated along (1, 1) direction
torch.manual_seed(0)
X = torch.randn(500, 2) @ torch.tensor([[2.0, 1.0], [0.0, 0.5]])

components, projected, ev = pca(X, k=2)
print("top components (each column is a direction):")
print(components)
print("explained variance:", ev.tolist())

**Three things worth saying out loud:**
1. **Centering matters.** Without it, you're finding directions of largest magnitude, not variance.
2. **Use SVD, not the eigendecomposition of `X.T @ X`.** Forming the covariance matrix loses precision when `D` is large or data is poorly conditioned.
3. **Whitening** = dividing the projection by the singular values, gives unit-variance components. One extra line if they ask.


---
# Pre-interview cheatsheet

## The five-line training step
```python
optimizer.zero_grad(set_to_none=True)
preds = model(x)
loss  = criterion(preds, y)
loss.backward()
optimizer.step()
```

## Tensor shape conventions
- **CNN**: `(B, C, H, W)`
- **Sequence / Transformer**: `(B, L, D)` with `batch_first=True`
- **RNN with `batch_first=False`** (default): `(L, B, D)`

## Common dtype gotchas
- Indices and class targets are `torch.long`, NOT `float`
- Masks for `masked_fill` are `torch.bool`
- Mixing `float16` and `float32` in a manual computation will silently downcast — use `autocast`

## Optimizer defaults worth knowing
- `Adam` / `AdamW`: `lr=1e-3`, `betas=(0.9, 0.999)`. AdamW decouples weight decay — almost always preferred over Adam.
- `SGD` with momentum: `lr=1e-2`, `momentum=0.9`. Standard for ResNets.

## When `view` vs `reshape`
- `view` requires contiguous memory. Fails after a `transpose` unless you `.contiguous()` first.
- `reshape` works either way — copies if needed. Slightly slower but safer.

## Three questions to ask if stuck
1. "Can I assume the input is on the same device as the model?"
2. "Should I assume the input shape is `(B, C, H, W)` or `(B, H, W, C)`?"
3. "Are there constraints I should consider — memory, latency, batch size?"

## Always write `assert` statements for shape constraints.

Each of these is an opening to demonstrate ML maturity, not weakness.


# FAQ

## What is the Vanishing Gradient Problem?

Backpropagation in neural networks works on the basis of the chain rule of differentiation.
- according to the chain rule, the gradient of the loss function with respect to the input layer parameters can be written as a product of gradients at each layer

If these gradients are all less than 1 (and worse still, heading toward 0), then the product of these gradients will be a vanishly small value

The Vanishing Gradient Problem can cause serious trouble in the optimization process by preventing the network parameters from changing their values, which is equivalent to stunted learning.


## Why AdamW over Adam for the optimizer?

`AdamW` decouples weight decay from the gradient update
- this is the standard for modern models (especially transformers) because it regularizes better than standard `Adam`

## Correct Activation for Transformers?

`LayerNorm` is the correct choice for transformers (normalizes across the feature dim per token).

## Pre-Layer Norm vs. Post-Layer Norm: Why Pre-Layer is always preferred

In Post-LN:

In [ ]:
x= self.norm1(x + self.attn(x, x, x))

The residual addition happens inside the LayerNorm.

On the backward pass, the gradient from later layers has to flow through the LayerNorm's normalization (and its rescaling) before it can reach the residual stream's earlier path. LayerNorm's Jacobian attenuates the gradient magnitude.

*The deeper you stack Post-LN blocks, the more attenuation — gradients to early layers shrink.*

In Pre-LN:

In [ ]:
h = self.norm1(x)
x = x + self.attn(h + h + h)

The residual is outside the norm.

The gradient now has a clean identity path back through every residual connection — ∂loss/∂x_earlier includes a direct +1 contribution from each residual.

Gradients to early layers stay healthy without needing warmup or a tiny LR.

This is exactly why every modern transformer (GPT-2 onward, LLaMA, ViT, you name it) uses Pre-LN.

What does `GradScaler` exist to do?

It multiplies the loss by a large factor before .`backward()` so that small gradients — gradients that would underflow to zero in fp16's tiny dynamic range — get pushed up into representable territory.
- Then `unscale_` divides them back before the optimizer step.
- The entire mechanism exists to fight `fp16` underflow.

In real code, with `bf16` you remove the scaler entirely and just call `loss.backward()` then `optimizer.step()`.